In [1]:
import sys

import torch

import pandas as pd

import pm4py

from config.feature_config import FeatureConfig
from config.ga_config import GAConfig

from utils.general_utils import set_stdout_to_file, set_seed
from utils.feature_utils import df_to_sequence_array

from model.preprocessor import PreprocessorArtifacts

from model.next_event_model import ProcessLSTM
from model.model_wrapper import ModelWrapper

from ga_search.search import CounterfactualGA

from process.engine import ProcessModelConstraintEngine
from process.experimenter import ExperimentHandler

### --- Load Dataset & Models ---

In [2]:
set_seed(seed=777)

In [3]:
df = pd.read_excel(
    "../../../data/bpic17.xlsx",
    engine="openpyxl",
    keep_default_na=False,
    dtype={
        "case:concept:name": "string",
        "concept:name": "string",
        "lifecycle:transition": "string",
        "org:resource": "string",
        "case:LoanGoal": "string",
        "case:ApplicationType": "string",
        "Accepted": "string",
        "Selected": "string",
        "case:RequestedAmount": "float32",
        "FirstWithdrawalAmount": "float32",
        "NumberOfTerms": "float32",
        "MonthlyCost": "float32",
        "CreditScore": "float32",
        "OfferedAmount": "float32",
        "time_delta": "float32",
    }
)

df["time:timestamp"] = pd.to_datetime(df["time:timestamp"])

In [4]:
df.head(20)

,case:concept:name,time:timestamp,Accepted,CreditScore,FirstWithdrawalAmount,MonthlyCost,NumberOfTerms,OfferedAmount,Selected,case:ApplicationType,case:LoanGoal,case:RequestedAmount,concept:name,lifecycle:transition,org:resource,time_delta
0,Application_1000086665,2016-08-03 15:57:21.673,NA,0.0,0.0,0.000000,0.0,0.0,NA,New credit,"Other, see explanation",5000.0,A_Create Application,complete,User_1,0.000000e+00
1,Application_1000086665,2016-08-03 15:57:21.734,NA,0.0,0.0,0.000000,0.0,0.0,NA,New credit,"Other, see explanation",5000.0,A_Submitted,complete,User_1,6.100000e-02
2,Application_1000086665,2016-08-03 15:58:28.299,NA,0.0,0.0,0.000000,0.0,0.0,NA,New credit,"Other, see explanation",5000.0,A_Concept,complete,User_1,6.656500e+01
3,Application_1000086665,2016-08-05 13:57:07.419,NA,0.0,0.0,0.000000,0.0,0.0,NA,New credit,"Other, see explanation",5000.0,A_Accepted,complete,User_5,1.655191e+05
4,Application_1000086665,2016-08-05 13:59:57.320,True,0.0,5000.0,241.279999,22.0,5000.0,False,New credit,"Other, see explanation",5000.0,O_Create Offer,complete,User_5,1.699010e+02
5,Application_1000086665,2016-08-05 13:59:58.162,NA,0.0,0.0,0.000000,0.0,0.0,NA,New credit,"Other, see explanation",5000.0,O_Created,complete,User_5,8.420000e-01
6,Application_1000086665,2016-08-05 14:01:23.264,NA,0.0,0.0,0.000000,0.0,0.0,NA,New credit,"Other, see explanation",5000.0,O_Sent (mail and online),complete,User_5,8.510200e+01
7,Application_1000086665,2016-08-05 14:01:23.288,NA,0.0,0.0,0.000000,0.0,0.0,NA,New credit,"Other, see explanation",5000.0,A_Complete,complete,User_5,2.400000e-02
8,Application_1000086665,2016-09-05 06:00:36.710,NA,0.0,0.0,0.000000,0.0,0.0,NA,New credit,"Other, see explanation",5000.0,A_Cancelled,complete,User_1,2.649554e+06
9,Application_1000086665,2016-09-05 06:00:36.829,NA,0.0,0.0,0.000000,0.0,0.0,NA,New credit,"Other, see explanation",5000.0,O_Cancelled,complete,User_1,1.190000e-01


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [6]:
feature_config = FeatureConfig.load(
    path = "../pretrained_models/"
)

In [7]:
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['Accepted', 'CreditScore', 'FirstWithdrawalAmount', 'MonthlyCost', 'NumberOfTerms', 'OfferedAmount', 'Selected', 'case:ApplicationType', 'case:LoanGoal', 'case:RequestedAmount', 'concept:name', 'lifecycle:transition', 'org:resource', 'time_delta']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
case:LoanGoal                  categorical    case     yes    ['Boat', 'Business goal', 'Car', ...]    N/A        data_derived        
case:ApplicationType           categorical    case     yes    ['Limit raise', 'New credit']            N/A        data_derived        
Accepted         

In [8]:
preprocessor_artifacts = PreprocessorArtifacts.load(
    path = "../pretrained_models/"
)

In [9]:
model = ProcessLSTM.load(
    path = "../pretrained_models/"
)

In [10]:
model_wrapper = ModelWrapper(
    model=model,
    preprocessor_artifacts=preprocessor_artifacts,
    device=device
)

### --- Process Constraints ---

In [11]:
engine = ProcessModelConstraintEngine.load(
    path = "../pretrained_models/"
)

In [12]:
engine.parallel_sets

[{'A_Incomplete', 'A_Validating', 'O_Returned'}]

In [13]:
engine.branching_sets

[{'A_Complete',
  'A_Incomplete',
  'A_Validating',
  'O_Create Offer',
  'O_Created',
  'O_Returned',
  'O_Sent (mail and online)'},
 {'O_Create Offer', 'O_Created', 'O_Sent (mail and online)'},
 {'A_Denied', 'O_Refused'}]

### --- Experiments Generation ---

In [14]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/bpic17-cf_seed777_experiments_ga_output.txt", console=False)

In [15]:
generator = ExperimentHandler(
    constraint_engine=engine,
)

In [16]:
exp_df_sin, metadata_sin = ExperimentHandler.load("../experiments/cf_generated_experiments_single_desired")
print("Mined using parameters:", metadata_sin["parameters"])

In [17]:
exp_df_mul, metadata_mul = ExperimentHandler.load("../experiments/cf_generated_experiments_multiple_desired")
print("Mined using parameters:", metadata_mul["parameters"])

### --- Counterfactuals ---

In [18]:
ga_config = GAConfig(
    w_distance=1.0,
    w_sparsity=1.0,
    w_margin=1.0,
    w_process_violation=1.0,
)
ga_config.validate()

cf_GA = CounterfactualGA(
    ga_config=ga_config,
    feature_config=feature_config,
    model_wrapper=model_wrapper
)

In [19]:
results_sin = generator.run_experiment_df(
    cf_method=cf_GA,
    technique="GA_single_desired_seed777",
    exp_df=exp_df_sin,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/500 [00:00<?, ?case/s]

In [20]:
results_sin

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,Application_509718181,7,1,0,0.437524,0.304460,0.570588,0.630208,0.117647,...,0.832584,0.117647,0.319104,0.470588,0.167619,0.395833,0.000000,0.000000,0.0,0.0
1,0,Application_1424170205,9,1,0,0.418647,0.281411,0.555882,0.593750,0.285714,...,0.701922,0.285714,0.166207,0.235294,0.097120,0.250000,0.000000,0.000000,0.0,0.0
2,0,Application_542586744,10,1,3,0.476099,0.278670,0.673529,0.629167,0.347826,...,0.973243,0.347826,0.250417,0.352941,0.147894,0.375000,0.000000,0.000000,0.0,0.0
3,0,Application_1690291723,11,1,2,0.258916,0.164891,0.352941,0.313542,0.840000,...,1.131378,0.840000,0.145545,0.235294,0.055796,0.145833,0.000000,0.952474,0.0,1.0
4,0,Application_135180486,12,1,3,0.301490,0.170628,0.432353,0.383333,0.757407,...,1.058634,0.111111,0.019651,0.000000,0.039302,0.062500,0.865372,0.865372,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
374,48,Application_317628520,22,1,8,0.307992,0.187413,0.428571,0.418627,0.621277,...,0.754529,0.617021,0.049273,0.028571,0.069974,0.088235,0.000000,0.962098,0.0,1.0
375,48,Application_2144131288,23,1,5,0.339288,0.237148,0.441429,0.448529,0.623469,...,0.543836,0.285714,0.097798,0.057143,0.138453,0.156863,0.003461,0.003461,0.0,0.0
376,48,Application_2057503198,24,1,6,0.365389,0.222206,0.508571,0.492647,0.514706,...,0.768139,0.607843,0.072061,0.085714,0.058408,0.088235,0.000000,0.975124,0.0,1.0
377,48,Application_727907012,25,1,3,0.455249,0.314784,0.595714,0.675980,0.018868,...,0.605328,0.000000,0.232779,0.285714,0.179845,0.372549,0.000000,0.000000,0.0,0.0


In [21]:
results_mul = generator.run_experiment_df(
    cf_method=cf_GA,
    technique="GA_multiple_desired_seed777",
    exp_df=exp_df_mul,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/500 [00:00<?, ?case/s]

In [22]:
results_mul

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,Application_1589000981,8,2,0,0.382005,0.164010,0.600000,0.383333,0.363636,...,0.546970,0.363636,0.100000,0.200000,0.000000,0.083333,0.0,0.480545,0.0,0.500
1,0,Application_1367573970,10,2,0,0.307094,0.214189,0.400000,0.358333,0.461538,...,0.644872,0.461538,0.100000,0.200000,0.000000,0.083333,0.0,0.487560,0.0,0.500
2,0,Application_707684890,11,2,0,0.352799,0.235598,0.470000,0.416667,0.500000,...,0.683333,0.500000,0.100000,0.200000,0.000000,0.083333,0.0,0.483278,0.0,0.500
3,0,Application_111867995,12,2,2,0.362496,0.254993,0.470000,0.433333,0.533333,...,0.801123,0.533333,0.101123,0.200000,0.002246,0.166667,0.0,0.489893,0.0,0.500
4,0,Application_82157828,13,2,3,0.327669,0.215337,0.440000,0.408333,0.562500,...,0.745833,0.562500,0.100000,0.200000,0.000000,0.083333,0.0,0.488468,0.0,0.500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
378,48,Application_1049158506,13,8,4,0.378597,0.235765,0.521429,0.428205,0.720000,...,1.068252,0.720000,0.143124,0.142857,0.143390,0.205128,0.0,0.860363,0.0,0.875
379,48,Application_151062598,13,8,4,0.350976,0.212666,0.489286,0.396154,0.714000,...,0.705016,0.600000,0.053734,0.071429,0.036039,0.051282,0.0,0.984756,0.0,1.000
380,48,Application_1205404110,13,8,4,0.345289,0.215578,0.475000,0.417949,0.720000,...,1.069993,0.720000,0.170506,0.214286,0.126726,0.179487,0.0,0.859208,0.0,0.875
381,48,Application_1167485975,13,8,4,0.348516,0.197032,0.500000,0.411538,0.707000,...,0.858234,0.460000,0.167464,0.142857,0.192072,0.230769,0.0,0.859139,0.0,0.875


### --- Cleanup ---

In [23]:
sys.stdout = original_stdout
log_file.close()